# MIRO AI -- Banc de comparaison de modèles locaux (Kaggle, gratuit)

**Objectif** : héberger `gpt-oss:20b` et `qwen3:14b` sur le GPU Kaggle et les exposer via un tunnel. Le banc de synthèse (`scripts/run_synthesis_model_bench.py replay`, `todo/contexte.txt` section 985) tourne sur TA machine et appelle cette URL.

**Confidentialité (ne pas contourner)** : ce notebook ne fait qu'héberger les modèles. Aucune donnée client ni instance Odoo réelle ne doit transiter par Kaggle, seulement le corpus de test du projet.

**Réglages de session** : `Settings` -> `Accelerator` = GPU T4 x2 (ou P100), `Internet` = ON.

**Poids des modèles : Dataset privé (une fois, ~22 Go)**
- **1re session (Mode A)** : les modèles sont téléchargés (cellule 5), puis sauvegardés dans un Dataset privé (cellule 9).
- **Sessions suivantes (Mode B)** : `+ Add Input` -> ton dataset. Plus aucun téléchargement.

**Secrets** (`Add-ons` -> `Secrets`) : `NGROK_AUTHTOKEN` (tunnel) ; `KAGGLE_API_TOKEN` (seulement pour créer le dataset).

**Quota** : 30 h GPU/semaine (reset le lundi), session ~9 h max ; garder la session active.

In [ ]:
# 1. GPU assigné (si aucun : Settings -> Accelerator, puis redémarrer la session)
!nvidia-smi

In [ ]:
# 2. Installer Ollama (zstd et pciutils absents de l'image Kaggle)
!apt-get -qq update && apt-get -qq install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 3. Configuration + détection du dataset (Mode A / Mode B)
import os, glob, subprocess

MODELS = ["gpt-oss:20b", "qwen3:14b"]
CONTEXT_LENGTH = 16384      # baisser (8192) si un modèle déborde sur le CPU (`ollama ps`, colonne PROCESSOR)
COPY_TO_LOCAL = True        # Mode B : copier le dataset sur le disque local avant usage (lecture mmap directe = risque de lenteur)
PULL_EMBED = False          # True seulement pour un run RAG complet
EMBED_MODEL = "nomic-embed-text"

LOCAL_MODELS_DIR = "/kaggle/temp/ollama_models"   # /kaggle/working est plafonné à 20 Go : trop petit pour 2 modèles

# Mode B : un dataset attaché contient un dossier `manifests` (structure d'un OLLAMA_MODELS)
_found = glob.glob("/kaggle/input/**/manifests", recursive=True)
if _found:
    DATASET_DIR = os.path.dirname(_found[0])
    MODE_B = True
    if COPY_TO_LOCAL:
        os.makedirs(LOCAL_MODELS_DIR, exist_ok=True)
        print(f"Mode B : copie de {DATASET_DIR} vers {LOCAL_MODELS_DIR} ...")
        subprocess.run(["rsync", "-a", "--info=progress2", DATASET_DIR + "/", LOCAL_MODELS_DIR + "/"], check=True)
        OLLAMA_MODELS_DIR = LOCAL_MODELS_DIR
    else:
        OLLAMA_MODELS_DIR = DATASET_DIR
    print(f"Mode B (dataset attaché) : modèles dans {OLLAMA_MODELS_DIR}")
else:
    MODE_B = False
    os.makedirs(LOCAL_MODELS_DIR, exist_ok=True)
    OLLAMA_MODELS_DIR = LOCAL_MODELS_DIR
    print(f"Mode A (pas de dataset) : téléchargement dans {OLLAMA_MODELS_DIR} ; sauvegarder ensuite via la cellule 9.")

os.environ["OLLAMA_MODELS"] = OLLAMA_MODELS_DIR
# Désactive le contrôle CORS Origin. Le rejet 403 du header Host est traité côté tunnel (host header réécrit).
os.environ["OLLAMA_ORIGINS"] = "*"
# Fenêtre par défaut = 4096 tokens (prompt + réponse) : coupe la synthèse multi-hop (Section 710).
os.environ["OLLAMA_CONTEXT_LENGTH"] = str(CONTEXT_LENGTH)
# Ne jamais décharger un modèle entre deux appels, mais un seul à la fois (VRAM).
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"
# Délai max de chargement d'un modèle (défaut 5 min).
os.environ["OLLAMA_LOAD_TIMEOUT"] = "30m"

In [ ]:
# 4. Serveur Ollama (rejouable : tue tout puis redémarre)
import time, requests

OLLAMA_URL = "http://localhost:11434"
LOG_PATH = "/kaggle/temp/ollama_serve.log"
ollama_proc = None

def stop_ollama():
    # -x : nom exact du process (évite que pkill se tue lui-même via `-f`) ; tue aussi les runners
    subprocess.run(["pkill", "-9", "-x", "ollama"])
    time.sleep(3)

def log_tail(n=40):
    try:
        return "".join(open(LOG_PATH, errors="replace").readlines()[-n:])
    except FileNotFoundError:
        return "(pas de log)"

def start_ollama():
    global ollama_proc
    stop_ollama()
    # Log dans un FICHIER, pas dans un pipe : un pipe jamais lu se remplit et bloque le serveur.
    ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=open(LOG_PATH, "w"),
                                   stderr=subprocess.STDOUT, env=os.environ)
    for _ in range(60):
        try:
            if requests.get(OLLAMA_URL, timeout=2).status_code == 200:
                print("Ollama prêt.")
                return
        except requests.exceptions.RequestException:
            pass
        time.sleep(1)
    raise RuntimeError("Ollama n'a pas démarré en 60 s. Log :\n" + log_tail())

def ollama_ps():
    r = subprocess.run(["ollama", "ps"], capture_output=True, text=True)
    return r.stdout + r.stderr

def diag():
    print("process :", "vivant" if ollama_proc and ollama_proc.poll() is None else f"mort (code {ollama_proc.poll() if ollama_proc else '?'})")
    try:
        requests.get(OLLAMA_URL, timeout=3)
        print("port 11434 : répond")
    except requests.exceptions.RequestException:
        print("port 11434 : ne répond pas")
    print(ollama_ps())
    print("--- fin du log serveur ---\n" + log_tail())

start_ollama()
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

In [ ]:
# 5. Modèles : téléchargement (Mode A) ou vérification (Mode B)
_wanted = MODELS + ([EMBED_MODEL] if PULL_EMBED else [])
if MODE_B:
    _installed = subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout
    print(_installed)
    _missing = [m for m in _wanted if m not in _installed]
    if _missing:
        print(f"ATTENTION : absent(s) du dataset : {_missing}. Repasser en Mode A (détacher le dataset), puis mettre à jour le dataset (cellule 9).")
else:
    for _m in _wanted:
        !ollama pull {_m}
    !ollama list

In [ ]:
# 6. Chargement + test de tool-calling, un modèle après l'autre
# Le payload inclut 'tools' : Ollama instancie un runner différent selon leur présence (Section 689).
WEATHER_TOOL = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Renvoie la météo pour une ville donnée.",
        "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]},
    },
}

def load_and_test(model, timeout=1800):
    t0 = time.time()
    try:
        r = requests.post(f"{OLLAMA_URL}/v1/chat/completions", timeout=timeout, json={
            "model": model, "stream": False, "tools": [WEATHER_TOOL],
            "messages": [{"role": "user", "content": "Quel temps fait-il à Paris ?"}],
        })
        r.raise_for_status()
        calls = r.json()["choices"][0]["message"].get("tool_calls")
        print(f"{model} : OK en {time.time() - t0:.0f} s, tool-calling {'OK' if calls else 'absent (non bloquant pour le banc de synthèse)'}")
    except Exception as e:
        print(f"{model} : ÉCHEC {type(e).__name__} après {time.time() - t0:.0f} s")
        diag()
        raise
    print(ollama_ps())  # PROCESSOR doit afficher 100% GPU

for _m in MODELS:
    load_and_test(_m)

## 7. Tunnel public

- **`ngrok` (défaut)** : secret `NGROK_AUTHTOKEN`. Pas de limite de durée par requête connue.
- **`cloudflared`** : aucun compte. Attention : les tunnels Cloudflare coupent en général une requête au bout de ~100 s ; une synthèse longue (gpt-oss avec raisonnement) peut dépasser (non vérifié ici).

In [ ]:
# 7. Tunnel vers Ollama
TUNNEL = "ngrok"   # "ngrok" ou "cloudflared"

if TUNNEL == "ngrok":
    !pip install -q pyngrok
    from pyngrok import ngrok, conf
    from kaggle_secrets import UserSecretsClient

    conf.get_default().auth_token = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
    for _t in ngrok.get_tunnels():
        ngrok.disconnect(_t.public_url)
    # host_header : Ollama rejette en 403 tout Host différent de localhost (Sections 682, 694).
    PUBLIC_URL = ngrok.connect(11434, "http", host_header="localhost:11434").public_url
else:
    import re, threading
    CLOUDFLARED_BIN = "/kaggle/temp/cloudflared"
    !curl -fsSL -o {CLOUDFLARED_BIN} https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x {CLOUDFLARED_BIN}
    cloudflared_proc = subprocess.Popen(
        [CLOUDFLARED_BIN, "tunnel", "--url", OLLAMA_URL, "--http-host-header", "localhost:11434"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    PUBLIC_URL = None
    _t0 = time.time()
    for _line in cloudflared_proc.stdout:
        _m = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", _line)
        if _m:
            PUBLIC_URL = _m.group(0)
            break
        if time.time() - _t0 > 60:
            break
    if not PUBLIC_URL:
        raise RuntimeError("URL cloudflared introuvable après 60 s.")
    # Le pipe cloudflared doit être lu en continu, sinon il se remplit et bloque le tunnel.
    threading.Thread(target=lambda: [None for _ in cloudflared_proc.stdout], daemon=True).start()

r = requests.get(f"{PUBLIC_URL}/api/tags", timeout=30)
print("Test via le tunnel : HTTP", r.status_code, "-", [m["name"] for m in r.json().get("models", [])])
print("\nURL publique (valable tant que ce notebook tourne) :")
print(PUBLIC_URL)

## 8. Côté de ta machine (pipeline MIRO AI + accès Odoo)

**Étape 1, `capture`** (une seule fois, depuis ta machine, sans ce notebook : LLM habituel + instance Odoo ; enregistre les prompts dans `captures/frozen_evidence/`) :

```bash
python scripts/run_synthesis_model_bench.py capture PUR-Q1 PUR-Q5 STK-Q1
```

**Étape 2, `replay`** (notebook actif, cellule 6 terminée). Remplacer `URL` par l'URL réelle, **sans chevrons**. Un modèle à la fois, une commande à la fois :

```bash
python scripts/run_synthesis_model_bench.py replay --base-url URL/v1 --model ollama:gpt-oss:20b --samples 3 --report-stem captures/bench_gptoss
python scripts/run_synthesis_model_bench.py replay --base-url URL/v1 --model ollama:qwen3:14b  --samples 3 --report-stem captures/bench_qwen3
```

Ne pas interrompre (Ctrl+C) un `replay` pendant le chargement d'un modèle : Ollama annule alors le chargement. Le notebook reçoit les prompts de synthèse gelés (données de l'instance de TEST, chemins absolus de ta machine).

## 9. Sauvegarder les poids dans un Dataset privé (une seule fois, en Mode A)

1. Créer un token sur https://www.kaggle.com/settings et le stocker dans le secret `KAGGLE_API_TOKEN`.
2. Renseigner `KAGGLE_USERNAME` ci-dessous et passer `SAVE_TO_KAGGLE_DATASET = True`.
3. Sessions suivantes : `+ Add Input` -> ce dataset (Mode B, cellule 3).

Pour mettre à jour le dataset (nouveau modèle) : Mode A, puis `kaggle datasets version` à la place de `create`.

In [ ]:
# 9. Push vers un Kaggle Dataset privé (désactivé par défaut)
SAVE_TO_KAGGLE_DATASET = False
KAGGLE_USERNAME = ""                      # ton pseudo Kaggle
DATASET_NAME = "ollama-models-miro"

if SAVE_TO_KAGGLE_DATASET:
    assert not MODE_B, "Déjà en Mode B : les modèles viennent d'un dataset."
    assert KAGGLE_USERNAME, "Renseigner KAGGLE_USERNAME."
    import json
    from kaggle_secrets import UserSecretsClient

    os.environ["KAGGLE_API_TOKEN"] = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
    stop_ollama()  # plus aucun accès aux fichiers pendant l'upload
    meta = {"title": DATASET_NAME, "id": f"{KAGGLE_USERNAME}/{DATASET_NAME}", "licenses": [{"name": "CC0-1.0"}]}
    json.dump(meta, open(os.path.join(OLLAMA_MODELS_DIR, "dataset-metadata.json"), "w"))
    !pip install -q kaggle
    !kaggle datasets create -p {OLLAMA_MODELS_DIR} --dir-mode zip
    print("Dataset créé (privé par défaut). L'attacher aux sessions suivantes via '+ Add Input'.")
    start_ollama()
else:
    print("SAVE_TO_KAGGLE_DATASET = False : rien à faire.")

In [ ]:
# En cas de blocage / timeout : diagnostic, puis (si le serveur est mort ou bloqué) redémarrage.
diag()
# start_ollama()   # décommenter pour redémarrer ; ensuite relancer la cellule 6 (rechargement des modèles)